
<img src="https://raw.githubusercontent.com/Vidhusv/autoMIC/main/logo.png" height="400" align="right" style="height:240px">


# autoMIC v1.0: Automated MIC Calculation from 96‑Well Plate Data


**Easy‑to‑use MIC determination for microbiology labs.**  
Upload your plate reader export (CSV/Excel), configure the layout, and instantly get:

- ✅ MIC value (µg/mL)  
- 📈 Dose‑response curve  
- 🗺️ Growth heatmap  
- 🧪 CLSI/EUCAST interpretation (sensitive / intermediate / resistant)  

No installation – runs entirely in your browser via Google Colab.  
All data stays on your machine; no uploads to external servers.

**How to cite:**  
If you use autoMIC in your research, please cite:

> Vidhu Smitha Vijay, Oudlin Mary Lenin. *autoMIC: a free, web‑based tool for automated MIC analysis.* Zenodo, 2025. DOI: [10.5281/zenodo.20590057](https://doi.org/10.5281/zenodo.20590057)

For updates, custom breakpoints, or to report issues:  
[https://github.com/Vidhusv/autoMIC](https://github.com/Vidhusv/autoMIC)

In [ ]:
# @title Install & import libraries
!pip install -q openpyxl ipywidgets matplotlib seaborn pandas numpy plotly

import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from google.colab import files

In [ ]:
# @title 1. Upload your plate reader file
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

def load_plate_data(file_name):
    """Read CSV or Excel, using the first column as row labels (A-H)."""
    if file_name.endswith('.csv'):
        df = pd.read_csv(file_name, index_col=0)
    else:
        df = pd.read_excel(file_name, index_col=0, sheet_name=0)
    # Convert all values to numeric (force errors to NaN)
    df = df.apply(pd.to_numeric, errors='coerce')
    # Expect 12 columns (wells 1-12)
    if df.shape[1] != 12:
        print(f"Warning: found {df.shape[1]} columns, expected 12. Heatmap may look odd.")
    return df

df = load_plate_data(file_name)
print("Loaded plate data shape:", df.shape)
df.head()

In [ ]:
# @title 2. Configure your plate and threshold (live-update ready)

# --- Live update toggle ---
live_update_checkbox = widgets.Checkbox(
    value=True,
    description='Auto‑refresh on change'
)

# --- Concentration for each row A–H (comma separated, µg/mL) ---
conc_input = widgets.Text(
    value='0,0.5,1,2,4,8,16,32',
    description='Conc. (µg/mL) A-H:',
    layout=widgets.Layout(width='80%')
)

# --- Negative control row (blank / no bacteria) ---
neg_control_row = widgets.Dropdown(
    options=['A','B','C','D','E','F','G','H'],
    value='A',
    description='Neg. control row:'
)

# --- Positive control row (bacteria + no antibiotic) – optional ---
pos_control_row = widgets.Dropdown(
    options=['None','A','B','C','D','E','F','G','H'],
    value='None',
    description='Pos. control row (optional):'
)

# --- Triplicate columns (well numbers, 1‑based) ---
col_start = widgets.IntText(value=1, description='Start col:')
col_end   = widgets.IntText(value=3, description='End col:')

# --- Inhibition threshold ---
threshold_type = widgets.RadioButtons(
    options=['Absolute OD ≤', 'Percent of positive control'],
    value='Absolute OD ≤',
    description='Threshold type:'
)
threshold_value = widgets.FloatText(value=0.1, step=0.01, description='Value:')

# --- Compute button (manual trigger, optional) ---
compute_btn = widgets.Button(description='Calculate MIC', button_style='success')

# --------------------------------------------------------------------
# LIVE‑UPDATE LOGIC – calls the compute_mic() function (defined later)
# --------------------------------------------------------------------
def _trigger_compute(change):
    """Re‑calculate MIC automatically if live‑update is enabled."""
    if live_update_checkbox.value and 'compute_mic' in globals():
        compute_mic(None)   # None = no button event

# Watch all relevant widgets for changes
for widget in [conc_input, neg_control_row, pos_control_row,
               threshold_type, threshold_value, col_start, col_end]:
    widget.observe(_trigger_compute, names='value')

# Also re‑calculate when live‑update checkbox is toggled
live_update_checkbox.observe(_trigger_compute, names='value')

# --------------------------------------------------------------------
# Display everything
# --------------------------------------------------------------------
display(widgets.VBox([
    live_update_checkbox,
    conc_input,
    neg_control_row,
    pos_control_row,
    threshold_type,
    threshold_value,
    widgets.HBox([col_start, col_end]),
    compute_btn
]))

In [ ]:
# @title 3. Run calculation & see results (with dose‑response download)

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output
from google.colab import files

# ---- Output area ----
output_area = widgets.Output()

# ---- Global variables ----
mic_result = None
summary_text = ""

# ---- Download button ----
download_dr_btn = widgets.Button(
    description='Download Dose‑Response PNG',
    button_style='info',
    icon='download'
)
def download_dose_response(b):
    try:
        files.download('dose_response.png')
    except Exception as e:
        print(f"Download failed: {e}")

download_dr_btn.on_click(download_dose_response)

# ---- Main calculation function ----
def compute_mic(b=None):
    global mic_result, summary_text

    with output_area:
        clear_output()

        # --- Parse concentrations ---
        conc_str = conc_input.value.strip()
        conc_list = [float(x) for x in conc_str.split(',')]
        if len(conc_list) != 8:
            print("Error: Please enter exactly 8 concentrations (rows A-H).")
            return

        rows = ['A','B','C','D','E','F','G','H']
        conc_map = {rows[i]: conc_list[i] for i in range(8)}

        neg_row = neg_control_row.value
        pos_row = pos_control_row.value
        if pos_row == 'None':
            pos_row = None

        # --- Triplicate columns ---
        start_col = col_start.value
        end_col = col_end.value
        col_names = [str(c) for c in range(start_col, end_col+1)]
        plate_cols = [str(c) for c in df.columns]
        matched_cols = []
        for c in col_names:
            if c in plate_cols:
                matched_cols.append(df.columns[plate_cols.index(c)])
            else:
                print(f"Column {c} not found in plate data. Available columns: {plate_cols}")
                return

        # --- Mean OD per row ---
        mean_ods = {}
        for row_label in rows:
            if row_label not in df.index:
                print(f"Row {row_label} not found.")
                return
            row_data = df.loc[row_label, matched_cols]
            mean_ods[row_label] = row_data.mean()

        # --- Corrected ODs ---
        neg_od = mean_ods.get(neg_row)
        if neg_od is None:
            print(f"Negative control row {neg_row} missing.")
            return
        corrected_ods = {r: mean_ods[r] - neg_od for r in rows}

        # --- Positive control ---
        pos_od = None
        if pos_row:
            if pos_row not in mean_ods:
                print(f"Positive control row {pos_row} missing.")
                return
            pos_od = mean_ods[pos_row] - neg_od

        # --- Threshold ---
        if threshold_type.value.startswith('Absolute'):
            threshold = threshold_value.value
        else:
            if pos_od is None or pos_od <= 0:
                print("Error: Need a valid positive control for percent threshold.")
                return
            threshold = (threshold_value.value / 100) * pos_od

        # --- MIC determination ---
        sorted_rows = sorted(rows, key=lambda r: conc_map[r])
        mic_conc = None
        for r in sorted_rows:
            if r == neg_row:
                continue
            if corrected_ods[r] <= threshold:
                mic_conc = conc_map[r]
                break

        # --- Numerical results ---
        print("Corrected mean ODs (blank subtracted):")
        for r in sorted_rows:
            print(f"  {r} ({conc_map[r]} µg/mL): {corrected_ods[r]:.4f}")
        print(f"\nThreshold: {threshold:.4f}")
        if mic_conc is not None:
            print(f"** MIC = {mic_conc} µg/mL **")
            summary_text = f"MIC = {mic_conc} µg/mL"
        else:
            print("No inhibition up to the highest concentration.")
            summary_text = "No MIC found"

        mic_result = mic_conc

        # --- Auto‑update interpretation ---
        try:
            refresh_interpretation()
        except NameError:
            pass

        # ============================================================
        #  MATPLOTLIB DOSE‑RESPONSE CURVE (saved as PNG automatically)
        # ============================================================
        x_vals = [conc_map[r] for r in sorted_rows]
        y_vals = [corrected_ods[r] for r in sorted_rows]

        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(x_vals, y_vals, 'o-', color='steelblue', linewidth=2, markersize=8, label='Corrected OD')
        ax.axhline(y=threshold, color='red', linestyle='--', linewidth=1.5, label=f'Threshold ({threshold})')

        # Highlight MIC
        if mic_conc is not None:
            idx = x_vals.index(mic_conc)
            ax.plot(mic_conc, y_vals[idx], marker='*', color='red', markersize=15, zorder=5)
            ax.annotate(f'MIC = {mic_conc} µg/mL',
                        xy=(mic_conc, y_vals[idx]),
                        xytext=(mic_conc + 0.5, y_vals[idx] + 0.02),
                        color='red', fontweight='bold',
                        arrowprops=dict(arrowstyle='->', color='red'))

        ax.set_xlabel('Concentration (µg/mL)')
        ax.set_ylabel('Corrected OD')
        ax.set_title('Dose‑Response Curve')
        ax.legend()
        ax.grid(alpha=0.3)
        plt.tight_layout()

        # ---- Save for download ----
        plt.savefig('dose_response.png', dpi=150, bbox_inches='tight')
        plt.show()
        plt.close()   # prevents duplicate display in some Colab versions

        # ============================================================
        #  STATIC HEATMAP (seaborn)
        # ============================================================
        plt.figure(figsize=(10, 4))
        plate_matrix = df.loc[rows, :].values
        annot = np.empty_like(plate_matrix, dtype=object)
        for i in range(plate_matrix.shape[0]):
            for j in range(plate_matrix.shape[1]):
                annot[i, j] = f"{plate_matrix[i, j]:.3f}"

        sns.heatmap(plate_matrix, annot=annot, fmt='', cmap='RdYlGn_r',
                    xticklabels=df.columns, yticklabels=rows,
                    cbar_kws={'label': 'OD'})
        plt.title("Static heatmap (green = low growth, red = high growth)")
        plt.tight_layout()
        plt.show()

        print("\n(Use the dropdown below for CLSI/EUCAST interpretation.)")

# ---- Wire up the manual button ----
compute_btn.on_click(compute_mic)

# ---- Run once on startup ----
compute_mic()

# ---- Display buttons and output ----
display(widgets.HBox([compute_btn, download_dr_btn]))
display(output_area)

In [ ]:
# @title 4. CLSI/EUCAST interpretation – LIVE, no button needed
organism_dropdown = widgets.Dropdown(
    options=['E. coli / Enterobacterales', 'Pseudomonas aeruginosa',
             'Staphylococcus aureus', 'Enterococcus spp.'],
    value='E. coli / Enterobacterales',
    description='Organism:'
)
antibiotic_dropdown = widgets.Dropdown(
    options=[],
    description='Antibiotic:',
    disabled=False
)

# =============================================================================
# BREAKPOINT LIBRARY (same as before)
# =============================================================================
breakpoints = {
    'E. coli / Enterobacterales': {
        'Ampicillin':                  {'S': 8,   'R': 32},
        'Amoxicillin-clavulanate':     {'S': 8,   'R': 32},
        'Piperacillin-tazobactam':     {'S': 16,  'R': 128},
        'Cefazolin':                   {'S': 2,   'R': 8},
        'Cefuroxime':                  {'S': 8,   'R': 32},
        'Ceftriaxone':                 {'S': 1,   'R': 4},
        'Ceftazidime':                 {'S': 4,   'R': 16},
        'Cefepime':                    {'S': 2,   'R': 16},
        'Meropenem':                   {'S': 1,   'R': 4},
        'Imipenem':                    {'S': 1,   'R': 4},
        'Ertapenem':                   {'S': 0.5, 'R': 2},
        'Ciprofloxacin':               {'S': 0.25,'R': 1},
        'Levofloxacin':                {'S': 0.5, 'R': 2},
        'Gentamicin':                  {'S': 4,   'R': 16},
        'Amikacin':                    {'S': 16,  'R': 64},
        'Tobramycin':                  {'S': 4,   'R': 16},
        'Trimethoprim-sulfamethoxazole':{'S': 2,  'R': 4},
        'Tetracycline':                {'S': 4,   'R': 16},
        'Tigecycline':                 {'S': 0.5, 'R': 2},
        'Colistin':                    {'S': 2,   'R': 4},
        'Nitrofurantoin':              {'S': 32,  'R': 128},
        'Fosfomycin (oral)':           {'S': 64,  'R': 256},
        'Ceftolozane-tazobactam':      {'S': 2,   'R': 8},
    },
    'Pseudomonas aeruginosa': {
        'Ceftazidime':                 {'S': 8,   'R': 32},
        'Cefepime':                    {'S': 8,   'R': 32},
        'Meropenem':                   {'S': 2,   'R': 8},
        'Imipenem':                    {'S': 2,   'R': 8},
        'Piperacillin-tazobactam':     {'S': 16,  'R': 128},
        'Ciprofloxacin':               {'S': 0.5, 'R': 2},
        'Levofloxacin':                {'S': 1,   'R': 4},
        'Gentamicin':                  {'S': 4,   'R': 16},
        'Amikacin':                    {'S': 16,  'R': 64},
        'Tobramycin':                  {'S': 4,   'R': 16},
        'Colistin':                    {'S': 2,   'R': 4},
        'Ceftolozane-tazobactam':      {'S': 4,   'R': 16},
    },
    'Staphylococcus aureus': {
        'Oxacillin (MSSA)':            {'S': 2,   'R': 4},
        'Cefoxitin (MRSA screen)':     {'S': 4,   'R': 8},
        'Vancomycin':                  {'S': 2,   'R': 8},
        'Daptomycin':                  {'S': 1,   'R': 4},
        'Linezolid':                   {'S': 4,   'R': 8},
        'Clindamycin':                 {'S': 0.5, 'R': 4},
        'Erythromycin':                {'S': 0.5, 'R': 8},
        'Gentamicin':                  {'S': 4,   'R': 16},
        'Ciprofloxacin':               {'S': 0.5, 'R': 2},
        'Levofloxacin':                {'S': 1,   'R': 4},
        'Trimethoprim-sulfamethoxazole':{'S': 2,  'R': 4},
        'Tetracycline':                {'S': 4,   'R': 16},
        'Rifampin':                    {'S': 1,   'R': 4},
    },
    'Enterococcus spp.': {
        'Ampicillin':                  {'S': 8,   'R': 16},
        'Vancomycin':                  {'S': 4,   'R': 32},
        'Linezolid':                   {'S': 2,   'R': 8},
        'Daptomycin':                  {'S': 4,   'R': 8},
        'Tetracycline':                {'S': 4,   'R': 16},
        'Nitrofurantoin':              {'S': 32,  'R': 128},
        'Ciprofloxacin':               {'S': 1,   'R': 4},
    },
}

# ---- Update antibiotic list when organism changes ----
def update_antibiotics(change):
    org = organism_dropdown.value
    abx_list = list(breakpoints.get(org, {}).keys())
    antibiotic_dropdown.options = abx_list
    if abx_list:
        antibiotic_dropdown.value = abx_list[0]

organism_dropdown.observe(update_antibiotics, 'value')
update_antibiotics(None)  # initialise

# ---- Output area (will live-update) ----
interp_output = widgets.Output()

# ---- Function to refresh interpretation ----
def refresh_interpretation(*args):
    """Called whenever organism, antibiotic, or MIC changes."""
    with interp_output:
        clear_output()
        if mic_result is None:
            print("MIC not calculated yet. Adjust settings to see results.")
            return
        org = organism_dropdown.value
        abx = antibiotic_dropdown.value
        bp = breakpoints.get(org, {}).get(abx)
        if bp is None:
            print(f"No breakpoints for {abx} in {org}.")
            return
        if mic_result <= bp['S']:
            cat = 'Sensitive '
        elif mic_result >= bp['R']:
            cat = 'Resistant '
        else:
            cat = 'Intermediate '
        print(f"**{org}**  |  {abx}  |  MIC = {mic_result} µg/mL  →  **{cat}**")

# ---- Watch for changes in dropdowns ----
organism_dropdown.observe(refresh_interpretation, 'value')
antibiotic_dropdown.observe(refresh_interpretation, 'value')

# ---- Also refresh when mic_result changes (hook into compute_mic later) ----
# We'll call refresh_interpretation() at the end of compute_mic in Cell 5.
# (Added below – no need to modify here, but we ensure function exists.)

# ---- Display widgets ----
display(widgets.VBox([
    organism_dropdown,
    antibiotic_dropdown,
    interp_output
]))

# Initial display (once mic_result exists)
if mic_result is not None:
    refresh_interpretation()

In [ ]:
# @title 5. Download a summary
download_btn = widgets.Button(description='Download Summary (.txt)')

def download_summary(b):
    if summary_text:
        with open('autoMIC_summary.txt', 'w') as f:
            f.write(summary_text + '\n')
        files.download('autoMIC_summary.txt')
    else:
        print("No results yet. Run the MIC calculation first.")

download_btn.on_click(download_summary)
display(download_btn)

# autoMIC User Guide

## 1. Quick Start

1. **Open the notebook in Google Colab** (File → Open notebook → GitHub / upload).
2. **Run all cells** (Runtime → Run all). This installs required libraries and displays the interface.
3. **Upload your plate reader file** (CSV or Excel) using the “Upload” button in Cell 1.
4. **Configure your plate** (Cell 2):
   - Enter the antibiotic concentration for each row A–H (comma‑separated, µg/mL).
   - Select the **negative control** row (blank / no bacteria).
   - Optionally select the **positive control** row (bacteria + no antibiotic).
   - Choose the triplicate columns (e.g., 1–3) that contain your replicates.
   - Set the **inhibition threshold** (absolute OD or % of positive control).
5. The MIC, dose‑response curve, and heatmap appear automatically (live mode).
6. Use the **CLSI/EUCAST interpretation** dropdowns (Cell 4) to see if the strain is Sensitive, Intermediate, or Resistant.

---

## 2. What Each Section Does

### Configuration Panel (Cell 2)
- **Auto‑refresh checkbox**: When ON, the tool recalculates instantly when you change any setting. Turn OFF if you want to adjust multiple things at once.
- **Concentration rows**: List the antibiotic concentrations in the **same order as rows A–H**. For example, `0,0.5,1,2,4,8,16,32` means row A = 0, B = 0.5, … H = 32.
- **Negative control**: The row with no bacteria (blank). All ODs are subtracted by its average to correct for background.
- **Positive control**: The row with bacteria but no antibiotic (optional). Used if you choose “Percent of positive control” as threshold.
- **Triplicate columns**: Typically columns 1–3. The tool averages these wells to give a mean OD for each concentration.
- **Threshold type**:
  - **Absolute OD ≤**: MIC is the lowest concentration where corrected OD falls below this value (e.g., 0.1).
  - **Percent of positive control**: MIC is the first concentration where growth is ≤ X% of the positive control (e.g., 10%).

### Results Tab (Cell 3)
- Shows the corrected ODs, threshold, and the calculated **MIC**.
- Displays a **dose‑response curve** (corrected OD vs concentration) with a star at the MIC and a red dashed threshold line.
- A **static heatmap** of the whole plate (green = low growth, red = high growth).
- A **Download Dose‑Response PNG** button saves the curve for reports.
- The interpretation section (Cell 4) updates live.

### Interpretation (Cell 4)
- Pick the **organism** and **antibiotic** from the dropdowns. The result updates automatically.
- Breakpoints are based on CLSI M100 (2024) and EUCAST v14.0. **Always verify with your local guidelines.**
- You can add or modify breakpoints by editing the dictionary inside Cell 4.

---

## 3. Expected Output

- **MIC**: The lowest concentration that inhibits growth. E.g., `MIC = 2.0 µg/mL`.
- **Dose‑response curve**: A line that drops sharply at the MIC – confirms the calculation is correct.
- **Heatmap**: Visual check that your triplicates are consistent and the blank (row A) has low OD.
- **Interpretation**: `Sensitive 🟢`, `Intermediate 🟡`, or `Resistant 🔴`.

---

## 4. Troubleshooting Common Errors

### ❌ “Error: Please enter exactly 8 concentrations”
**Cause**: The concentration field is empty or has the wrong number of values.  
**Fix**: Enter exactly 8 numbers separated by commas (e.g., `0,0.5,1,2,4,8,16,32`).

### ❌ “Column X not found in plate data”
**Cause**: The triplicate start/end columns don’t exist in your uploaded file.  
**Fix**: Check your file’s column headers. Usually they are 1,2,3…12. Adjust `Start col` and `End col` to match (e.g., 1 and 3 for triplicates in wells 1–3).

### ❌ “Row X not found”
**Cause**: The uploaded file does not have rows labelled A–H.  
**Fix**: Ensure the first column of your CSV/Excel contains the letters A, B, … H (not numbers or different labels).

### ❌ “Negative control row missing” or “Positive control row missing”
**Cause**: The selected control row doesn’t exist in the data.  
**Fix**: Choose a row that actually has data (A–H). For positive control, set to “None” if you don’t have one.

### ❌ “Please run the MIC calculation first”
**Cause**: You tried to get an interpretation before any data was processed.  
**Fix**: Upload a file and run the calculation cell first. It runs automatically on start, but if you restarted the runtime you may need to re‑upload.

### ❌ Dose‑response curve or heatmap is blank
**Cause**: Rare Colab rendering glitch.  
**Fix**: Try clicking the “Calculate MIC” button. If it persists, restart the runtime (Runtime → Restart runtime) and run all cells again.

### ❌ Interpretation shows “No breakpoints for …”
**Cause**: The chosen antibiotic/organism combination is not in the built‑in library.  
**Fix**: You can add your own breakpoints. Edit the `breakpoints` dictionary in Cell 4 – see **Customisation** below.

---

## 5. Customising the Breakpoints

Open **Cell 4** and find the `breakpoints = { ... }` dictionary.  
Each organism group has a list of antibiotics with `'S'` and `'R'` values (µg/mL).  
To add a new antibiotic, simply copy an existing line and change the name and values.

Make sure to preserve the commas. If you need to add a completely new organism group, copy the entire block for one organism and modify.

Important: The breakpoints provided are examples from CLSI/EUCAST. Always confirm with your local reference standard before using for clinical decisions.

---

## 6. Downloading Your Results
Dose‑response PNG: Click the blue “Download Dose‑Response PNG” button. The file is saved as dose_response.png every time the MIC is calculated.

Summary text: If you need a plain text file with the MIC value, use the download button provided in the original notebook (if included) or simply copy the MIC output.

---

## 7. Still Stuck?
Check the GitHub repository for updates and FAQs: https://github.com/Vidhusv/autoMIC

Open an issue on GitHub describing your problem. Include:

The exact error message.

A screenshot of your configuration panel.

(Optional) The first few rows of your plate reader file (anonymised if needed).

Contact the developers:
Vidhu Smitha Vijay & Oudlin Mary Lenin
[https://github.com/Vidhusv]

---

## 8. Citation
If you use autoMIC in your work, please cite:

Vidhu Smitha Vijay, Oudlin Mary Lenin. autoMIC: a free, web‑based tool for automated MIC analysis. Zenodo, 2025. DOI: [10.5281/zenodo.20590057]